In [1]:
import pandas as pd
import numpy as np

In [2]:
weather_df = pd.read_csv("datasets/wind_turbine_clusters_hourly_features/on_n33_weather_test.csv")


In [3]:
print(weather_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43848 entries, 0 to 43847
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  43848 non-null  object 
 1   wind_speed_100m       43848 non-null  float64
 2   wind_direction_100m   43848 non-null  float64
 3   wind_speed_10m        43848 non-null  float64
 4   wind_direction_10m    43848 non-null  float64
 5   wind_gusts_10m        43848 non-null  float64
 6   surface_pressure      43848 non-null  float64
 7   temperature_2m        43848 non-null  float64
 8   cloud_cover_low       43848 non-null  float64
 9   cloud_cover_mid       43848 non-null  float64
 10  cloud_cover_high      43848 non-null  float64
 11  relative_humidity_2m  43848 non-null  float64
 12  rain                  43848 non-null  float64
dtypes: float64(12), object(1)
memory usage: 4.3+ MB
None


In [4]:
#converting date column into datatime object for feature extraction
weather_df['date'] = pd.to_datetime(weather_df['date']) 

In [5]:
years = weather_df['date'].dt.year 
print(years)

0        2020
1        2020
2        2020
3        2020
4        2020
         ... 
43843    2025
43844    2025
43845    2025
43846    2025
43847    2025
Name: date, Length: 43848, dtype: int32


In [6]:
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43848 entries, 0 to 43847
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   date                  43848 non-null  datetime64[ns]
 1   wind_speed_100m       43848 non-null  float64       
 2   wind_direction_100m   43848 non-null  float64       
 3   wind_speed_10m        43848 non-null  float64       
 4   wind_direction_10m    43848 non-null  float64       
 5   wind_gusts_10m        43848 non-null  float64       
 6   surface_pressure      43848 non-null  float64       
 7   temperature_2m        43848 non-null  float64       
 8   cloud_cover_low       43848 non-null  float64       
 9   cloud_cover_mid       43848 non-null  float64       
 10  cloud_cover_high      43848 non-null  float64       
 11  relative_humidity_2m  43848 non-null  float64       
 12  rain                  43848 non-null  float64       
dtypes: datetime64[ns

In [7]:
# Extract temporal components
date_source = weather_df['date'].dt
weather_df['hour'] = date_source.hour
weather_df['day_of_week'] = date_source.dayofweek
weather_df['month'] = date_source.month
weather_df['day_of_year'] = date_source.dayofyear

In [8]:
# Apply cyclical encoding for time features
# Hour
weather_df['hour_sin'] = np.sin(2 * np.pi * weather_df['hour'] / 24.0)
weather_df['hour_cos'] = np.cos(2 * np.pi * weather_df['hour'] / 24.0)
# Day of week
weather_df['day_of_week_sin'] = np.sin(2 * np.pi * weather_df['day_of_week'] / 7.0)
weather_df['day_of_week_cos'] = np.cos(2 * np.pi * weather_df['day_of_week'] / 7.0)
# Month
weather_df['month_sin'] = np.sin(2 * np.pi * (weather_df['month'] - 1) / 12.0) # Month is 1-12
weather_df['month_cos'] = np.cos(2 * np.pi * (weather_df['month'] - 1) / 12.0)
# Day of year
weather_df['day_of_year_sin'] = np.sin(2 * np.pi * (weather_df['day_of_year'] - 1) / 366.0) # Use 366 for divisor
weather_df['day_of_year_cos'] = np.cos(2 * np.pi * (weather_df['day_of_year'] - 1) / 366.0)

In [9]:
print("\nDataFrame head after adding encoded temporal features:")
# Display only relevant columns to keep output concise
print(weather_df[['date', 'hour', 'hour_sin', 'hour_cos', 'month', 'month_sin', 'month_cos']].head())
print(f"\nNumber of columns after adding temporal features: {len(weather_df.columns)}")



DataFrame head after adding encoded temporal features:
                 date  hour  hour_sin  hour_cos  month  month_sin  \
0 2020-04-01 00:00:00     0  0.000000  1.000000      4        1.0   
1 2020-04-01 01:00:00     1  0.258819  0.965926      4        1.0   
2 2020-04-01 02:00:00     2  0.500000  0.866025      4        1.0   
3 2020-04-01 03:00:00     3  0.707107  0.707107      4        1.0   
4 2020-04-01 04:00:00     4  0.866025  0.500000      4        1.0   

      month_cos  
0  6.123234e-17  
1  6.123234e-17  
2  6.123234e-17  
3  6.123234e-17  
4  6.123234e-17  

Number of columns after adding temporal features: 25


In [10]:
# Apply cyclical encoding for wind direction features

# Wind Direction 100m
col_100m = 'wind_direction_100m'
if col_100m in weather_df.columns:
    rad_100m = np.deg2rad(weather_df[col_100m]) # Convert degrees to radians
    weather_df[f'{col_100m}_sin'] = np.sin(rad_100m)
    weather_df[f'{col_100m}_cos'] = np.cos(rad_100m)
    print(f"Encoded '{col_100m}'.")
else:
    print(f"Warning: Column '{col_100m}' not found for encoding.")


Encoded 'wind_direction_100m'.


In [11]:
col_10m = 'wind_direction_10m'
if col_10m in weather_df.columns:
    rad_10m = np.deg2rad(weather_df[col_10m]) # Convert degrees to radians
    weather_df[f'{col_10m}_sin'] = np.sin(rad_10m)
    weather_df[f'{col_10m}_cos'] = np.cos(rad_10m)
    print(f"Encoded '{col_10m}'.")
else:
    print(f"Warning: Column '{col_10m}' not found for encoding.")

Encoded 'wind_direction_10m'.


In [12]:
    print(f"\nNumber of columns after adding wind direction features: {len(weather_df.columns)}")


Number of columns after adding wind direction features: 29


In [13]:
print(weather_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43848 entries, 0 to 43847
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     43848 non-null  datetime64[ns]
 1   wind_speed_100m          43848 non-null  float64       
 2   wind_direction_100m      43848 non-null  float64       
 3   wind_speed_10m           43848 non-null  float64       
 4   wind_direction_10m       43848 non-null  float64       
 5   wind_gusts_10m           43848 non-null  float64       
 6   surface_pressure         43848 non-null  float64       
 7   temperature_2m           43848 non-null  float64       
 8   cloud_cover_low          43848 non-null  float64       
 9   cloud_cover_mid          43848 non-null  float64       
 10  cloud_cover_high         43848 non-null  float64       
 11  relative_humidity_2m     43848 non-null  float64       
 12  rain                     43848 n

In [14]:
cols_to_drop = [
    'hour', 'day_of_week', 'month', 'day_of_year', # Original integer time components
]


In [15]:
if 'wind_direction_100m_sin' in weather_df.columns:
    cols_to_drop.append('wind_direction_100m')
if 'wind_direction_10m_sin' in weather_df.columns:
    cols_to_drop.append('wind_direction_10m')


In [16]:
print(weather_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43848 entries, 0 to 43847
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     43848 non-null  datetime64[ns]
 1   wind_speed_100m          43848 non-null  float64       
 2   wind_direction_100m      43848 non-null  float64       
 3   wind_speed_10m           43848 non-null  float64       
 4   wind_direction_10m       43848 non-null  float64       
 5   wind_gusts_10m           43848 non-null  float64       
 6   surface_pressure         43848 non-null  float64       
 7   temperature_2m           43848 non-null  float64       
 8   cloud_cover_low          43848 non-null  float64       
 9   cloud_cover_mid          43848 non-null  float64       
 10  cloud_cover_high         43848 non-null  float64       
 11  relative_humidity_2m     43848 non-null  float64       
 12  rain                     43848 n

In [17]:
weather_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print("\nDropped original non-encoded temporal and direction columns.")
print(f"\nFinal number of columns: {len(weather_df.columns)}")
print("\nColumns remaining:")
print(weather_df.columns)



Dropped original non-encoded temporal and direction columns.

Final number of columns: 23

Columns remaining:
Index(['date', 'wind_speed_100m', 'wind_speed_10m', 'wind_gusts_10m',
       'surface_pressure', 'temperature_2m', 'cloud_cover_low',
       'cloud_cover_mid', 'cloud_cover_high', 'relative_humidity_2m', 'rain',
       'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos',
       'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos',
       'wind_direction_100m_sin', 'wind_direction_100m_cos',
       'wind_direction_10m_sin', 'wind_direction_10m_cos'],
      dtype='object')


In [18]:
# Display final DataFrame head and info
print("\nFinal Processed DataFrame head:")
print(weather_df.head())
print("\nFinal Processed DataFrame info:")
weather_df.info()


Final Processed DataFrame head:
                 date  wind_speed_100m  wind_speed_10m  wind_gusts_10m  \
0 2020-04-01 00:00:00        12.287555        6.379216       10.440001   
1 2020-04-01 01:00:00        17.227419        8.534353       14.040000   
2 2020-04-01 02:00:00        20.870687        9.885262       16.919998   
3 2020-04-01 03:00:00        23.683512       11.681987       19.800000   
4 2020-04-01 04:00:00        24.703976       11.923557       20.519999   

   surface_pressure  temperature_2m  cloud_cover_low  cloud_cover_mid  \
0         1026.9287         -0.1565              4.0              0.0   
1         1026.1285         -0.0565             28.0              0.0   
2         1025.3278          0.6435             34.0              0.0   
3         1024.8278          0.7935             42.0              0.0   
4         1024.0277          0.6435             49.0              2.0   

   cloud_cover_high  relative_humidity_2m  ...  day_of_week_sin  \
0              8

In [69]:
energy_df= pd.read_csv("datasets/on_and_off_shore_actual_energy_generation_2017_2025/on_and_off_shore_actual_energy_generation_2017_2018-Copy1.csv")

In [71]:
print(energy_df.info())
print(energy_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35044 entries, 0 to 35043
Data columns (total 5 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Area                                    35044 non-null  object 
 1   MTU                                     35044 non-null  object 
 2   Solar - Actual Aggregated [MW]          35040 non-null  float64
 3   Wind Offshore - Actual Aggregated [MW]  35040 non-null  float64
 4   Wind Onshore - Actual Aggregated [MW]   35040 non-null  float64
dtypes: float64(3), object(2)
memory usage: 1.3+ MB
None
     Area                                             MTU  \
0  BZN|NL  01.01.2017 00:00 - 01.01.2017 00:15 (CET/CEST)   
1  BZN|NL  01.01.2017 00:15 - 01.01.2017 00:30 (CET/CEST)   
2  BZN|NL  01.01.2017 00:30 - 01.01.2017 00:45 (CET/CEST)   
3  BZN|NL  01.01.2017 00:45 - 01.01.2017 01:00 (CET/CEST)   
4  BZN|NL  01.01.2017 01:00 - 01.01.2017 01:15 

In [73]:
#Parse Start Time from MTU Column
energy_df['start_time_str'] = energy_df['MTU'].str.split(' - ', expand=True)[0]

In [75]:
print(energy_df[['MTU', 'start_time_str']].head())

                                              MTU    start_time_str
0  01.01.2017 00:00 - 01.01.2017 00:15 (CET/CEST)  01.01.2017 00:00
1  01.01.2017 00:15 - 01.01.2017 00:30 (CET/CEST)  01.01.2017 00:15
2  01.01.2017 00:30 - 01.01.2017 00:45 (CET/CEST)  01.01.2017 00:30
3  01.01.2017 00:45 - 01.01.2017 01:00 (CET/CEST)  01.01.2017 00:45
4  01.01.2017 01:00 - 01.01.2017 01:15 (CET/CEST)  01.01.2017 01:00


In [77]:
local_tz = 'Europe/Amsterdam' 

In [79]:
# Convert the start time string to naive datetime objects
naive_datetime = pd.to_datetime(energy_df['start_time_str'], format='%d.%m.%Y %H:%M')

In [81]:
# Localize the naive datetime to the local timezone (CET/CEST)
# Handle nonexistent times (like during summer time, where time goes forward 1h) by shifting them forward to the next valid time
local_datetime = naive_datetime.dt.tz_localize(local_tz, ambiguous='infer', nonexistent='shift_forward')

In [83]:
#Convert the localized datetime to UTC
energy_df['Timestamp (UTC)'] = local_datetime.dt.tz_convert('UTC')

In [85]:
print("\nDataFrame head after timezone conversion (with nonexistent handling):")
# Show original string, naive, localized, and UTC versions for comparison
temp_df_view = pd.DataFrame({
    'start_str': energy_df['start_time_str'],
    'naive': naive_datetime,
    'local_CET_CEST': local_datetime,
    'UTC': energy_df['Timestamp (UTC)']
})


DataFrame head after timezone conversion (with nonexistent handling):


In [87]:
print(temp_df_view.head())

          start_str               naive            local_CET_CEST  \
0  01.01.2017 00:00 2017-01-01 00:00:00 2017-01-01 00:00:00+01:00   
1  01.01.2017 00:15 2017-01-01 00:15:00 2017-01-01 00:15:00+01:00   
2  01.01.2017 00:30 2017-01-01 00:30:00 2017-01-01 00:30:00+01:00   
3  01.01.2017 00:45 2017-01-01 00:45:00 2017-01-01 00:45:00+01:00   
4  01.01.2017 01:00 2017-01-01 01:00:00 2017-01-01 01:00:00+01:00   

                        UTC  
0 2016-12-31 23:00:00+00:00  
1 2016-12-31 23:15:00+00:00  
2 2016-12-31 23:30:00+00:00  
3 2016-12-31 23:45:00+00:00  
4 2017-01-01 00:00:00+00:00  


In [89]:
energy_df.set_index('Timestamp (UTC)', inplace=True)
energy_df.sort_index(inplace=True) # Ensure data is sorted by time

In [91]:
print(energy_df.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 35044 entries, 2016-12-31 23:00:00+00:00 to 2017-12-31 22:45:00+00:00
Data columns (total 6 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Area                                    35044 non-null  object 
 1   MTU                                     35044 non-null  object 
 2   Solar - Actual Aggregated [MW]          35040 non-null  float64
 3   Wind Offshore - Actual Aggregated [MW]  35040 non-null  float64
 4   Wind Onshore - Actual Aggregated [MW]   35040 non-null  float64
 5   start_time_str                          35044 non-null  object 
dtypes: float64(3), object(3)
memory usage: 1.9+ MB
None


In [93]:
column_rename_map = {
    'Wind Offshore - Actual Aggregated [MW]': 'Wind_Offshore_MW',
    'Wind Onshore - Actual Aggregated [MW]': 'Wind_Onshore_MW',
    'Solar - Actual Aggregated [MW]': 'Solar_MW' 
}
energy_df.rename(columns=column_rename_map, inplace=True)

In [95]:
# Select only the relevant columns (our targets + solar for later)
energy_targets_df = energy_df[['Wind_Offshore_MW', 'Wind_Onshore_MW', 'Solar_MW']].copy()

print("\nDataFrame head after setting index, renaming and selecting columns:")
print(energy_targets_df.head())
print("\nInfo after setting index and selecting columns:")
energy_targets_df.info()


DataFrame head after setting index, renaming and selecting columns:
                           Wind_Offshore_MW  Wind_Onshore_MW  Solar_MW
Timestamp (UTC)                                                       
2016-12-31 23:00:00+00:00             206.0            349.0       0.0
2016-12-31 23:15:00+00:00             204.0            409.0       0.0
2016-12-31 23:30:00+00:00             207.0            414.0       0.0
2016-12-31 23:45:00+00:00             206.0            405.0       0.0
2017-01-01 00:00:00+00:00             203.0            396.0       0.0

Info after setting index and selecting columns:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 35044 entries, 2016-12-31 23:00:00+00:00 to 2017-12-31 22:45:00+00:00
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Wind_Offshore_MW  35040 non-null  float64
 1   Wind_Onshore_MW   35040 non-null  float64
 2   Solar_MW          35040 non-null  flo

In [98]:
# Resample to hourly frequency ('h'), calculating the mean MW for each hour
energy_targets_hourly_df = energy_targets_df.resample('h').mean()

print("\nDataFrame head after resampling to hourly:")
print(energy_targets_hourly_df.head())
print("\nInfo after resampling:")
energy_targets_hourly_df.info() # Note the change in number of rows



DataFrame head after resampling to hourly:
                           Wind_Offshore_MW  Wind_Onshore_MW  Solar_MW
Timestamp (UTC)                                                       
2016-12-31 23:00:00+00:00            205.75           394.25       0.0
2017-01-01 00:00:00+00:00            208.25           387.75       0.0
2017-01-01 01:00:00+00:00            219.00           381.75       0.0
2017-01-01 02:00:00+00:00            221.25           396.25       0.0
2017-01-01 03:00:00+00:00            223.00           413.25       0.0

Info after resampling:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8760 entries, 2016-12-31 23:00:00+00:00 to 2017-12-31 22:00:00+00:00
Freq: h
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Wind_Offshore_MW  8760 non-null   float64
 1   Wind_Onshore_MW   8760 non-null   float64
 2   Solar_MW          8760 non-null   float64
dtypes: float64(3)
memory usage: 273.8

In [102]:
# Check for missing
print(energy_targets_hourly_df.isnull().sum())


Wind_Offshore_MW    0
Wind_Onshore_MW     0
Solar_MW            0
dtype: int64


In [110]:
print("\nFinal Processed Hourly Energy DataFrame head:")
print(energy_targets_hourly_df.head())
print("\nFinal Processed Hourly Energy DataFrame info:")
energy_targets_hourly_df.info()



Final Processed Hourly Energy DataFrame head:
                           Wind_Offshore_MW  Wind_Onshore_MW  Solar_MW
Timestamp (UTC)                                                       
2016-12-31 23:00:00+00:00            205.75           394.25       0.0
2017-01-01 00:00:00+00:00            208.25           387.75       0.0
2017-01-01 01:00:00+00:00            219.00           381.75       0.0
2017-01-01 02:00:00+00:00            221.25           396.25       0.0
2017-01-01 03:00:00+00:00            223.00           413.25       0.0

Final Processed Hourly Energy DataFrame info:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8760 entries, 2016-12-31 23:00:00+00:00 to 2017-12-31 22:00:00+00:00
Freq: h
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Wind_Offshore_MW  8760 non-null   float64
 1   Wind_Onshore_MW   8760 non-null   float64
 2   Solar_MW          8760 non-null   float64
dtypes: floa

In [ ]:
energy_targets_hourly_df.drop('Solar_MW', axis=1, inplace=True)

In [120]:
output_filename = 'energy_data_hourly_processed.csv' 
energy_targets_hourly_df.to_csv(output_filename, index=True)

In [123]:
output_filename = 'weather_data_processed.csv' 
weather_df.to_csv(output_filename, index=True)